# SVO descriptor variant benchmark

Source: archived SVO notebook, cells 18–38. The original and radius-4 maps are evaluated on the same reactions. The archived OOD cell overwrote its split; this notebook uses an explicit diene holdout.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from _modeling_common import (PHYSORG_FEATURE_COUNT, evaluate_targets,
                              load_aligned)

MAPS = {
    "Original SVO": DESCRIPTOR_MAP,
    "Radius-4 SVO": REPO_ROOT / "Data" / "Descriptors" / "Descriptors_SVO_2.0.pickle",
}
for path in MAPS.values():
    if not path.is_file():
        raise FileNotFoundError(path)
keys = ["Reaction_Index", "Diene_Index", "Ene_Index", "Structure_id"]
aligned = {name: load_aligned(DATA_DIR / "Active_Energy.csv", path)
           for name, path in MAPS.items()}
common_keys = set.intersection(*[
    set(map(tuple, frame[keys].to_numpy())) for frame, _ in aligned.values()
])
common = {}
for name, (frame, matrix) in aligned.items():
    selected = np.array([tuple(row) in common_keys for row in frame[keys].to_numpy()])
    subset, features = frame.loc[selected].reset_index(drop=True), matrix[selected]
    order = np.lexsort([subset[key].to_numpy() for key in reversed(keys)])
    common[name] = (subset.iloc[order].reset_index(drop=True), features[order])
if len({len(frame) for frame, _ in common.values()}) != 1:
    raise RuntimeError("The SVO maps do not share the same reactions")
print(f"Common reactions: {len(next(iter(common.values()))[0]):,}")

In [ ]:
comparison_rows = []
for map_name, (frame, features) in common.items():
    for fold, (train_ids, test_ids) in enumerate(
            KFold(5, shuffle=True, random_state=0).split(features)):
        for variant, matrix in [("PhysOrg", features[:, :PHYSORG_FEATURE_COUNT]),
                                ("PhysOrg + SVO", features)]:
            rows, _ = evaluate_targets(frame, matrix, train_ids, test_ids)
            comparison_rows.extend({"map": map_name, "split": "random CV",
                                    "fold": fold, "variant": variant, **row}
                                   for row in rows)
comparison = pd.DataFrame(comparison_rows)
display(comparison.groupby(["map", "variant", "target"])[["r2", "mae"]].mean())

## Diene-held-out comparison


In [ ]:
ood_rows = []
for map_name, (frame, features) in common.items():
    for seed in [0, 3, 6]:
        rng = np.random.default_rng(seed)
        diene_ids = rng.permutation(frame["Diene_Index"].unique())
        held_out = set(diene_ids[:max(1, len(diene_ids) // 10)])
        mask = frame["Diene_Index"].isin(held_out).to_numpy()
        train_ids, test_ids = np.flatnonzero(~mask), np.flatnonzero(mask)
        for variant, matrix in [("PhysOrg", features[:, :PHYSORG_FEATURE_COUNT]),
                                ("PhysOrg + SVO", features)]:
            rows, _ = evaluate_targets(frame, matrix, train_ids, test_ids)
            ood_rows.extend({"map": map_name, "split": "diene OOD",
                             "seed": seed, "variant": variant, **row} for row in rows)
ood_comparison = pd.DataFrame(ood_rows)
display(ood_comparison.groupby(["map", "variant", "target"])[["r2", "mae"]].mean())